# NF-v3 temporal graph construction

This notebook is intentionally thin. The tested graph builder lives in `code/python/scripts/build_nfv3_graphs.py`; this notebook only mounts Drive, configures paths, runs a small smoke build, and inspects its audit.

Run the smoke build first. Do not run the full build until its generated schema, mappings, provenance, and audit have been reviewed.

In [1]:
# ============================================================
# SETUP - Run this cell first
# ============================================================
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)], check=True)

sys.path.append(str(REPO_ROOT / 'code/python'))

Cloning into 'temporalgnn-nids'...
remote: Enumerating objects: 1273, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 1273 (delta 21), reused 52 (delta 20), pack-reused 1215 (from 2)
Receiving objects: 100% (1273/1273), 12.12 MiB | 20.51 MiB/s, done.
Resolving deltas: 100% (471/471), done.


In [2]:
!pip install -q torch-geometric

from google.colab import drive
drive.mount('/content/drive')

import json

PROJECT_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
CORRECTED_ROOT = PROJECT_ROOT / 'corrected_data' / 'infiltration_v1'
CORRECTED_CSV = CORRECTED_ROOT / 'nfv3_corrected.csv'
CORRECTED_MANIFEST = CORRECTED_ROOT / 'nfv3_corrected.manifest.json'

assert REPO_ROOT.is_dir(), f'Repository not found: {REPO_ROOT}'
assert CORRECTED_CSV.is_file(), f'Corrected CSV not found: {CORRECTED_CSV}'
assert CORRECTED_MANIFEST.is_file(), f'Corrected manifest not found: {CORRECTED_MANIFEST}'

GRAPH_VERSION = 'infiltration_v1_w30'
PREFLIGHT_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_preflight'
SMOKE_ROOT = PROJECT_ROOT / 'graphs' / f'{GRAPH_VERSION}_smoke'
FULL_ROOT = PROJECT_ROOT / 'graphs' / GRAPH_VERSION
PROFILES = ('nfv3_extended', 'portable_core')

print(f'Corrected CSV: {CORRECTED_CSV}')
print(f'Preflight output: {PREFLIGHT_ROOT}')
print(f'Smoke output: {SMOKE_ROOT}')
print(f'Full output: {FULL_ROOT}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 74.7 MB/s eta 0:00:00
Mounted at /content/drive
Corrected CSV: /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv
Preflight output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_preflight
Smoke output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_smoke
Full output: /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30


In [3]:
preflight_command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(PREFLIGHT_ROOT),
    '--chunksize', '250000',
    '--preflight-only',
    '--overwrite',
]
for profile in PROFILES:
    preflight_command += ['--profile', profile]

print(' '.join(preflight_command))
subprocess.run(preflight_command, check=True)

preflight = json.loads((PREFLIGHT_ROOT / 'feature_preflight.json').read_text())
print('Preflight status:', preflight['status'])
print('Input rows / positives:', preflight['input_rows'], preflight['positive_rows'])
print('Retained rows / positives:', preflight['retained_rows'], preflight['retained_positive_rows'])
print('Excluded rows / positives:', preflight['excluded_rows'], preflight['excluded_positive_rows'])
print('Counts by source file:', json.dumps(preflight['by_source_file'], indent=2))
print('Invalid endpoint rows (union):', preflight['invalid_any_endpoint_rows'])
print('Invalid endpoint positive rows:', preflight['invalid_any_endpoint_positive_rows'])
print('Invalid source endpoint rows:', preflight['invalid_source_endpoint_rows'])
print('Invalid destination endpoint rows:', preflight['invalid_destination_endpoint_rows'])
print('Invalid source endpoint reasons:', preflight['invalid_source_endpoint_reasons'])
print('Invalid destination endpoint reasons:', preflight['invalid_destination_endpoint_reasons'])
print('Invalid ports:', preflight['invalid_port_rows'])
print('Invalid protocols:', preflight['invalid_protocol_rows'])
print('Invalid binary targets:', preflight['invalid_binary_target_rows'])
print('Invalid time/duration rows:', preflight['invalid_time_or_duration_rows'])
print('Flow-end differences >1 ms:', preflight['flow_end_difference_gt_1ms_rows'])
print('Maximum absolute flow-end difference (ms):', preflight['flow_end_max_absolute_difference_ms'])
print('Invalid numeric rows:', preflight['invalid_numeric_rows_by_profile'])
print('Port categories:', preflight['port_category_counts'])
print('Port zero by protocol:', preflight['port_zero_by_protocol'])
print('Top other privileged ports:', preflight['other_privileged_top_ports'])
print('Top other high ports:', preflight['other_high_top_ports'])

python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_preflight --chunksize 250000 --preflight-only --overwrite --profile nfv3_extended --profile portable_core
Preflight status: passed
Invalid ports: 0
Invalid protocols: 0
Invalid numeric rows: {'nfv3_extended': 0, 'portable_core': 0}
Port categories: {'dst_port_admin_remote': 950413, 'dst_port_database': 3764, 'dst_port_infrastructure': 1543228, 'dst_port_not_applicable_or_zero': 14085, 'dst_port_other_high': 453250, 'dst_port_other_privileged': 22671, 'dst_port_web_http_proxy': 887789, 'dst_port_windows_smb_rpc': 305060}
Port zero by protocol: {'1': 6551, '2': 3740, '47': 4, '58': 3790}
Top other privileged ports

In [4]:
command = [
    'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
    '--input-csv', str(CORRECTED_CSV),
    '--corrected-manifest', str(CORRECTED_MANIFEST),
    '--output-root', str(SMOKE_ROOT),
    '--chunksize', '250000',
    '--max-windows', '10',
    '--overwrite',
]
for profile in PROFILES:
    command += ['--profile', profile]

print(' '.join(command))
subprocess.run(command, check=True)

python /content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py --input-csv /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv --corrected-manifest /content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json --output-root /content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_smoke --chunksize 250000 --max-windows 10 --overwrite --profile nfv3_extended --profile portable_core


CompletedProcess(args=['python', '/content/temporalgnn-nids/code/python/scripts/build_nfv3_graphs.py', '--input-csv', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.csv', '--corrected-manifest', '/content/drive/MyDrive/nids-fair-retrain/corrected_data/infiltration_v1/nfv3_corrected.manifest.json', '--output-root', '/content/drive/MyDrive/nids-fair-retrain/graphs/infiltration_v1_w30_smoke', '--chunksize', '250000', '--max-windows', '10', '--overwrite', '--profile', 'nfv3_extended', '--profile', 'portable_core'], returncode=0)

In [5]:
audit = json.loads((SMOKE_ROOT / 'graph_audit.json').read_text())
configuration = json.loads((SMOKE_ROOT / 'build_configuration.json').read_text())

print('Audit status:', audit['status'])
print('Day-1 cutoffs:', configuration['day1_cutoffs'])
print('Profile schema hashes:')
for name, digest in configuration['profiles'].items():
    print(f'- {name}: {digest}')

for day_name in ('day1', 'day2'):
    mapping = json.loads((SMOKE_ROOT / 'mappings' / f'{day_name}_ip_to_id.json').read_text())
    print(f"{day_name}: {mapping['entries']} mapped IPs; policy={mapping['creation_policy']}")

Audit status: partial
Day-1 cutoffs: {'raw_train_end_ms': 1519836776999, 'raw_val_end_ms': 1519849633500, 'train_end_ms': 1519836780000, 'val_end_ms': 1519849650000}
Profile schema hashes:
- nfv3_extended: e0c9659696cdd4260266f871f7a73d59ab86aaa8b9e0fd0094569193f1320236
- portable_core: 3b7b1e49d2ec46613016a454633428d1de31b7d0b5a416554819c1d6640f5cda
day1: 27 mapped IPs; policy=append_only_first_valid_chronological_appearance
day2: 35 mapped IPs; policy=append_only_first_valid_chronological_appearance


In [6]:
import pandas as pd
import torch

sample_provenance = next((SMOKE_ROOT / 'provenance' / 'day1').glob('graph_*.csv'))
sample_graph = SMOKE_ROOT / 'nfv3_extended' / 'train' / f'{sample_provenance.stem}.pt'
provenance = pd.read_csv(sample_provenance)
graph = torch.load(sample_graph, weights_only=False)

print('Graph:', sample_graph.name)
print('Nodes:', graph.num_nodes)
print('Edges:', graph.edge_index.shape[1])
print('Edge feature shape:', tuple(graph.edge_attr.shape))
print('Decision time (UTC epoch ms):', graph.timestamp)
display(provenance.head())

Graph: graph_1519776780000.pt
Nodes: 2
Edges: 1
Edge feature shape: (1, 33)
Decision time (UTC epoch ms): 1519776780000


,flow_id,source_file,source_row_id,source_ip,destination_ip,source_global_id,destination_global_id,edge_position,flow_start_ms,flow_end_ms,decision_time_ms,window_start_ms,window_end_ms,binary_target
0,cicids2018v3_wed2802.csv:0,cicids2018v3_wed2802.csv,0,5.188.9.25,172.31.64.89,0,1,0,1.519777e+12,1.519777e+12,1519776780000,1519776750000,1519776780000,0


## Full build

Run this cell only after reviewing the smoke-build output. Use a new versioned output directory instead of overwriting a reviewed graph collection. If Colab disconnects, rerun the same command with `--resume`.

In [ ]:
# full_command = [
#     'python', str(REPO_ROOT / 'code/python/scripts/build_nfv3_graphs.py'),
#     '--input-csv', str(CORRECTED_CSV),
#     '--corrected-manifest', str(CORRECTED_MANIFEST),
#     '--output-root', str(FULL_ROOT),
#     '--chunksize', '250000',
# ]
# for profile in PROFILES:
#     full_command += ['--profile', profile]
# subprocess.run(full_command, check=True)

# To resume an interrupted full build, append '--resume' to full_command and run it again.